# Retail Demand Forecasting: M5 Baseline

**Business question:** How many units should each store expect to sell during the next 28 days?

This editable notebook downloads authorized competition data, audits quality, creates a leakage-free time split, and compares forecasting baselines.

## 1. Setup
Accept the M5 competition rules before downloading. Never commit an API token to GitHub.

In [ ]:
# Uncomment in a fresh environment.
# %pip install -r ../requirements-dev.txt

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print(PROJECT_ROOT.resolve())

## 2. Download through Kaggle API
Outside Kaggle, create an API token at `https://www.kaggle.com/settings/api`. Set `KAGGLE_API_TOKEN` as an environment variable or run `kagglehub.login()` when prompted.

In [ ]:
from download_data import download
download(DATA_DIR)  # Safe to rerun; existing files are reused.

## 3. Data audit

In [ ]:
from data_audit import build_audit
audit = build_audit(DATA_DIR)
pd.DataFrame(audit['files']).T[['rows', 'columns', 'size_mb']]

In [ ]:
pd.Series(audit['quality_checks'], name='value').to_frame()

## 4. Leakage-free 28-day backtest
Days 1–1,913 are training data. The next 28 days stay unseen until evaluation. Edit the forecast definitions to test new ideas without changing the holdout.

In [ ]:
HORIZON, TRAIN_END = 28, 1913
train_cols = [f'd_{day}' for day in range(1, TRAIN_END + 1)]
test_cols = [f'd_{day}' for day in range(TRAIN_END + 1, TRAIN_END + HORIZON + 1)]
sales = pd.read_csv(DATA_DIR / 'sales_train_evaluation.csv', usecols=train_cols + test_cols)
train = sales[train_cols].to_numpy(dtype=np.float32)
actual = sales[test_cols].to_numpy(dtype=np.float32)
train.shape, actual.shape

In [ ]:
def score(actual, predicted, train):
    error = actual - predicted
    scale = np.mean(np.diff(train, axis=1) ** 2, axis=1)
    usable = scale > 0
    return {'MAE': np.abs(error).mean(), 'WAPE': np.abs(error).sum() / np.abs(actual).sum(), 'RMSSE': np.sqrt(np.mean(error[usable] ** 2, axis=1) / scale[usable]).mean(), 'Bias': error.sum() / np.abs(actual).sum()}

forecasts = {
    'Last value': np.repeat(train[:, -1:], HORIZON, axis=1),
    'Mean · last 28 days': np.repeat(train[:, -28:].mean(axis=1, keepdims=True), HORIZON, axis=1),
    'Seasonal · 7 days': np.tile(train[:, -7:], (1, 4)),
    'Seasonal · 28 days': train[:, -28:].copy(),
}
results = pd.DataFrame([{'Model': name, **score(actual, prediction, train)} for name, prediction in forecasts.items()])
results.sort_values('WAPE')

In [ ]:
fig = px.bar(results.sort_values('WAPE'), x='WAPE', y='Model', orientation='h', text_auto='.1%')
fig.update_layout(title='28-day baseline comparison', xaxis_tickformat='.0%')
fig.show()

## 5. Executive conclusion
The 28-day moving average is the benchmark to beat. The next experiment adds lag features, rolling demand, price changes, events, product identity and store identity to a global LightGBM model. A model is useful only if it improves the unchanged holdout and supports a better inventory decision.